# Sea Surface Temperature Forecasting with FNO2D

Forecast future ocean surface temperature fields from past ones with a 2D Fourier Neural Operator
trained on NOAA Optimum Interpolation Sea Surface Temperature (OISST) data. The model learns the
operator

$$G: T(\cdot, t) \mapsto T(\cdot, t + \Delta t)$$

that advances the temperature field $T$ by one time step.

In [ ]:
# Install keras-sciml, then choose a backend: "tensorflow", "torch" or "jax"
!pip install keras-sciml
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

## Hyperparameters

`TIME_HISTORY` past fields are stacked as input channels and `TIME_FUTURE` fields are predicted. Set `USE_SYNTHETIC = True` to train on physically-motivated synthetic fields instead of downloading OISST data. `N_SAMPLES` and `RESOLUTION` can be set to subsample the data for a faster run.

In [ ]:
N_SAMPLES = None
RESOLUTION = None
TIME_HISTORY = 1
TIME_FUTURE = 1
USE_SYNTHETIC = False
MODES = 8
WIDTH = 24
DEPTH = 3
EPOCHS = 25
BATCH_SIZE = 16
LEARNING_RATE = 0.005

## Load the data

`load_sea_surface_temperature` returns standardised train/test pairs. If the download fails, it falls
back to synthetic fields, and `meta["source"]` says which was used.

In [ ]:
import keras
import numpy as np

from keras_sciml.datasets import load_sea_surface_temperature
from keras_sciml.losses import RelativeL2Loss, relative_l2_error
from keras_sciml.models import FNO2D

keras.utils.set_random_seed(42)

x_train, y_train, x_test, y_test, meta = load_sea_surface_temperature(
    download=True,
    n_samples=N_SAMPLES,
    resolution=RESOLUTION,
    time_history=TIME_HISTORY,
    time_future=TIME_FUTURE,
    step_mode="pairs",
    train_ratio=0.8,
    normalize=True,
    use_synthetic=USE_SYNTHETIC,
    seed=42,
)
print(f"Data source: {meta['source']}")
print(f"Train: {x_train.shape}, Test: {x_test.shape}")
print(f"Mean SST: {meta['mean']:.2f} deg C, Std: {meta['std']:.2f} deg C")

## Define the model

In [ ]:
model = FNO2D(
    modes=(MODES, MODES),
    width=WIDTH,
    depth=DEPTH,
    out_channels=TIME_FUTURE,
    activation="gelu",
)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=RelativeL2Loss(),
    metrics=[RelativeL2Loss(name="rel_l2"), "mae"],
)

## Train

In [ ]:
history = model.fit(
    x_train,
    y_train,
    validation_data=(x_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=2,
)

## Evaluate

The MAE is converted back to degrees Celsius using the standard deviation from standardisation.

In [ ]:
predictions = keras.ops.convert_to_numpy(model.predict(x_test, verbose=0))
y_test_np = keras.ops.convert_to_numpy(y_test)

test_rel_l2 = float(np.mean(keras.ops.convert_to_numpy(relative_l2_error(y_test_np, predictions))))
mae_celsius = float(np.mean(np.abs(y_test_np - predictions)) * meta["std"])
print(f"Test relative L2 error: {test_rel_l2:.4f}")
print(f"Test mean absolute error: {mae_celsius:.3f} deg C")